# **공정 최적화 알고리즘**

- 참고 : https://chatgpt.com/share/69149855-f564-8013-a0aa-68f3246e6a8b

- **문제**
<img src="https://drive.google.com/uc?export=view&id=1X1FlY_tdOKZlVNpDPOkOLwV5-u-Hssfq" width="60%" title="세탁 공정 최적화 문제">


✅ 최종 해답(요약)

- 세탁기 대수(주어짐): 6대
- 목표 라인 처리량(세탁 기준): 18 kg/시간 ((10+50)/1hr,  3kg/시간/기계 --> 3 x 1 x 6 =18kg/시간
- 필요 건조기 대수: <mark>**3대**</mark>  (4kg/40분 → 6kg/시간/대, 3대 x 6 = **18 kg/시간**)
- 인원 수(최소)
    - 세탁 단계: <mark>**1명**</mark>  (10분 준비/60분 x 6대 = 60분/시간 → 1인분/시간)
    - 건조 단계: <mark>**1명**</mark>  (5분 준비/40분 x 3대 = 22.5분/시간)
    - 다림질 단계: <mark>**4명**</mark> (한 사람 기준 4kg 배치 처리: 준비 10분 + 1kg당 10분x4 = 총 50분 → 4.8kg/시간/인 → 18/4.8 ≈ 3.75 → 반올림↑ 4명)

## **Solution**

In [1]:
# ---------------------------------------------
# [문제 명세 정리]
# - 목적: 세탁소를 24시간/휴식없이 운영할 때, 라인이 막히지 않도록
#   (1) 건조기 대수, (2) 각 단계 인원 수(최소)를 산출
# - 제약: 전 단계 완료 후 다음 단계 시작, '작업준비'는 인력 필요
# ---------------------------------------------

from math import ceil

# ---------------------------------------------
# [문제의 이해 → 공정 파라미터 정의]
#  - 각 단계는 (작업준비시간, 기계/작업시간, 배치 용량[kg])로 모델링
#  - 세탁/건조는 '기계' 시간이 있고, 다림질은 '인력 작업'으로 본다.
# ---------------------------------------------

class StageParams:
    def __init__(self, setup_min, process_min, batch_kg):
        self.setup_min = setup_min      # 작업준비(인력) 시간/배치
        self.process_min = process_min  # 기계 동작 or 인력 작업 시간/배치
        self.batch_kg = batch_kg        # 1배치가 처리하는 무게(kg)

# 문제에서 주어진 값들
WASHER_COUNT = 6
wash = StageParams(setup_min=10, process_min=50, batch_kg=3)  # 3kg/건, 10+50=60분
dry  = StageParams(setup_min=5,  process_min=35, batch_kg=4)  # 4kg/건, 5+35=40분
iron = StageParams(setup_min=10, process_min=10, batch_kg=4)  # 다림질은 4kg 배치로 묶어 계산(건조 출력과 동기화)

# ---------------------------------------------
# [설계 방향 결정]
#  - 라인의 병목을 없애려면 각 단계의 시간당 처리량(kg/h)을 맞춘다.
#  - 세탁은 6대로 고정 → 세탁 기준 처리량을 '목표 처리량'으로 설정.
# ---------------------------------------------

def machine_throughput_per_hour(params: StageParams, machine_count: int) -> float:
    """(알고리즘 설계) 기계 단계 처리량 계산: kg/h = (배치kg) * (기계 수) / (사이클시간[h])"""
    cycle_h = (params.setup_min + params.process_min) / 60.0
    return params.batch_kg * machine_count / cycle_h

def human_setup_load_minutes_per_hour(params: StageParams, machine_count: int) -> float:
    """(알고리즘 설계) 해당 기계 수를 100% 돌릴 때 필요한 인력의 '작업준비' 총 시간(분/시간)"""
    # 1대당 시간당 배치 시작 횟수 = 1 / cycle_h
    cycle_h = (params.setup_min + params.process_min) / 60.0
    starts_per_hour_per_machine = 1.0 / cycle_h
    return params.setup_min * starts_per_hour_per_machine * machine_count

def people_needed_from_minutes_per_hour(minutes_per_hour: float) -> int:
    """(알고리즘 설계) 분/시간 → 인원(최소 정수, 1인=60분/시간 가정)"""
    return max(1, ceil(minutes_per_hour / 60.0))

# ---------------------------------------------
# [정확성 검증 아이디어]
#  - 세탁 처리량을 기준으로 목표 처리량을 구하고,
#    건조/다림질이 그 이상 처리 가능하도록 최소 설비/인력을 산출한다.
#  - 간단한 시뮬레이션 관점의 sanity check: 결과 처리량 비교 출력.
# ---------------------------------------------

# 세탁 라인의 목표 처리량(kg/h)
wash_line_tph = machine_throughput_per_hour(wash, WASHER_COUNT)

# 세탁 인원(작업준비 전담) 산출
wash_setup_min_per_h = human_setup_load_minutes_per_hour(wash, WASHER_COUNT)
wash_people = people_needed_from_minutes_per_hour(wash_setup_min_per_h)

# 건조기 최소 대수 산출: 건조 라인이 wash_line_tph 이상 처리하도록
def min_dryers_needed(target_tph: float, params: StageParams) -> int:
    # 1대 처리량
    per_machine_tph = machine_throughput_per_hour(params, 1)
    return ceil(target_tph / per_machine_tph)

dryers = min_dryers_needed(wash_line_tph, dry)

# 건조 인원(작업준비 전담) 산출
dry_setup_min_per_h = human_setup_load_minutes_per_hour(dry, dryers)
dry_people = people_needed_from_minutes_per_hour(dry_setup_min_per_h)

# 다림질 인원 산출: 다림질은 '인력 공정'으로 1인 처리량을 계산하여 배치
def ironing_tph_per_person(params: StageParams) -> float:
    """(알고리즘 설계) 다림질 1인당 처리량(kg/h) = 배치kg / (준비+작업 시간[h])"""
    cycle_h = (params.setup_min + params.process_min * params.batch_kg) / 60.0
    # 주의: 다림질은 1kg당 작업 10분 → 배치 전체 작업시간 = 10분 * 배치kg
    return params.batch_kg / cycle_h

iron_tph_one = ironing_tph_per_person(iron)
iron_people = ceil(wash_line_tph / iron_tph_one)

# ---------------------------------------------
# [알고리즘 분석(간단)]
#  - 시간 복잡도: O(1) (단순 산술식)
#  - 설비/인력 산출은 선형 탐색/반올림 수준
# ---------------------------------------------

# ---------------------------------------------
# [구현 결과 출력 + 간단 검증]
# ---------------------------------------------
def pretty(num, nd=2):
    return f"{num:.{nd}f}"

print("=== 공정 파라미터 요약 ===")
print(f"- 세탁: 준비 {wash.setup_min}분, 기계 {wash.process_min}분, 배치 {wash.batch_kg}kg")
print(f"- 건조: 준비 {dry.setup_min}분,  기계 {dry.process_min}분,  배치 {dry.batch_kg}kg")
print(f"- 다림질: 준비 {iron.setup_min}분, 작업 1kg당 {iron.process_min}분, 배치 {iron.batch_kg}kg 기준")

print("\n=== 세탁 기준 목표 처리량 ===")
print(f"- 세탁기 대수: {WASHER_COUNT}대")
print(f"- 세탁 처리량: {pretty(wash_line_tph)} kg/시간")
print(f"- 세탁 준비 인력 소요: {pretty(wash_setup_min_per_h)} 분/시간 → 최소 인원: {wash_people}명")

print("\n=== 건조 설비/인력 산출 ===")
print(f"- 필요 건조기 대수: {dryers}대")
print(f"- 건조 처리량(해당 대수): {pretty(machine_throughput_per_hour(dry, dryers))} kg/시간")
print(f"- 건조 준비 인력 소요: {pretty(dry_setup_min_per_h)} 분/시간 → 최소 인원: {dry_people}명")

print("\n=== 다림질 인력 산출 ===")
print(f"- 다림질 1인 처리량: {pretty(iron_tph_one)} kg/시간")
print(f"- 필요 다림질 인원: {iron_people}명")


=== 공정 파라미터 요약 ===
- 세탁: 준비 10분, 기계 50분, 배치 3kg
- 건조: 준비 5분,  기계 35분,  배치 4kg
- 다림질: 준비 10분, 작업 1kg당 10분, 배치 4kg 기준

=== 세탁 기준 목표 처리량 ===
- 세탁기 대수: 6대
- 세탁 처리량: 18.00 kg/시간
- 세탁 준비 인력 소요: 60.00 분/시간 → 최소 인원: 1명

=== 건조 설비/인력 산출 ===
- 필요 건조기 대수: 3대
- 건조 처리량(해당 대수): 18.00 kg/시간
- 건조 준비 인력 소요: 22.50 분/시간 → 최소 인원: 1명

=== 다림질 인력 산출 ===
- 다림질 1인 처리량: 4.80 kg/시간
- 필요 다림질 인원: 4명


## **연습 과제**(선택)

1. **세탁기를 8대로 늘리면** 필요한 **건조기·다림질 인원**은 어떻게 변할까?
2. **다림질을 3kg 배치**로만 한다면(세탁과 동기화) **인원**은 몇 명이 필요할까?
3. **건조기의 준비시간을 3분으로 줄이는 개선안**이 있다면 **설비/인력**에 어떤 영향을 줄까?

In [2]:
# =============================================
# 공통 유틸 & 모델 (문제해결 절차에 맞춘 구조)
# =============================================
from math import ceil
import pandas as pd

class StageParams:
    """각 공정 파라미터: 작업준비(인력), 공정시간, 배치용량(kg)"""
    def __init__(self, setup_min, process_min, batch_kg):
        self.setup_min = setup_min
        self.process_min = process_min
        self.batch_kg = batch_kg

def machine_throughput_per_hour(params: StageParams, machine_count: int) -> float:
    """[알고리즘 설계] 기계 공정 처리량(kg/h) = 배치kg * 대수 / (준비+공정 시간[h])
    (세탁·건조: process_min은 배치 전체 시간)"""
    cycle_h = (params.setup_min + params.process_min) / 60.0
    return params.batch_kg * machine_count / cycle_h

def human_setup_load_minutes_per_hour(params: StageParams, machine_count: int) -> float:
    """[알고리즘 설계] 해당 대수를 100% 운전할 때 필요한 작업준비 총량(분/시간)"""
    cycle_h = (params.setup_min + params.process_min) / 60.0
    starts_per_hour_per_machine = 1.0 / cycle_h
    return params.setup_min * starts_per_hour_per_machine * machine_count

def people_needed_from_minutes_per_hour(minutes_per_hour: float) -> int:
    """[알고리즘 설계] 분/시간 → 최소 인원(정수, 1인=60분/시간)"""
    return max(1, ceil(minutes_per_hour / 60.0))

def ironing_tph_per_person(setup_min: int, per_kg_min: int, batch_kg: int) -> float:
    """[알고리즘 설계] 다림질 1인 처리량(kg/h)
    - 준비시간은 배치당 1회, 작업시간은 1kg당 per_kg_min 분 * 배치kg"""
    cycle_h = (setup_min + per_kg_min * batch_kg) / 60.0
    return batch_kg / cycle_h

def solve_line(washers:int, wash_params:StageParams, dry_params:StageParams,
               iron_setup_min:int, iron_perkg_min:int, iron_batch_kg:int):
    """[구현] 한 번의 시나리오를 계산하고 표로 요약"""
    # 세탁 기준 처리량
    wash_tph = machine_throughput_per_hour(wash_params, washers)
    wash_setup_min_ph = human_setup_load_minutes_per_hour(wash_params, washers)
    wash_people = people_needed_from_minutes_per_hour(wash_setup_min_ph)

    # 건조기 최소 대수: 세탁 처리량 이상 달성
    per_dryer_tph = machine_throughput_per_hour(dry_params, 1)
    dryers = ceil(wash_tph / per_dryer_tph)
    dry_tph = machine_throughput_per_hour(dry_params, dryers)
    dry_setup_min_ph = human_setup_load_minutes_per_hour(dry_params, dryers)
    dry_people = people_needed_from_minutes_per_hour(dry_setup_min_ph)

    # 다림질 인원: 1인 처리량으로 커버
    iron_tph_one = ironing_tph_per_person(iron_setup_min, iron_perkg_min, iron_batch_kg)
    iron_people = ceil(wash_tph / iron_tph_one)

    df = pd.DataFrame([
        ["세탁", f"{washers}대", f"{wash_params.setup_min} + {wash_params.process_min} (분)",
         f"{wash_params.batch_kg} kg", f"{wash_tph:.2f}", f"{wash_people}명"],
        ["건조", f"{dryers}대", f"{dry_params.setup_min} + {dry_params.process_min} (분)",
         f"{dry_params.batch_kg} kg", f"{dry_tph:.2f}", f"{dry_people}명"],
        ["다림질", f"-", f"준비 {iron_setup_min} + 작업 {iron_perkg_min}×배치kg (분)",
         f"{iron_batch_kg} kg", f"{iron_tph_one:.2f} (1인)", f"{iron_people}명"],
    ], columns=["공정","설비대수","사이클(준비+작업)","배치용량","처리량 kg/h","필요 인원"])

    return {
        "wash_tph": wash_tph, "dryers": dryers, "dry_tph": dry_tph,
        "iron_tph_one": iron_tph_one, "wash_people": wash_people,
        "dry_people": dry_people, "iron_people": iron_people, "table": df
    }

def show(df):
    try:
        from caas_jupyter_tools import display_dataframe_to_user
        display_dataframe_to_user("라인 밸런싱 요약", df)
    except Exception:
        display(df)

print("✅ 준비 완료. 아래 셀에서 파라미터만 바꾸어 실행하세요.")

✅ 준비 완료. 아래 셀에서 파라미터만 바꾸어 실행하세요.


In [3]:
# =============================================
# 기본 파라미터 (문제에서 주어진 값)
# - 필요 시 이 값들을 수정하여 바로 실험 가능
# =============================================
WASHERS = 6
wash = StageParams(setup_min=10, process_min=50, batch_kg=3)
dry  = StageParams(setup_min=5,  process_min=35, batch_kg=4)
IRON_SETUP_MIN = 10       # 다림질 준비시간(분)
IRON_PERKG_MIN = 10       # 다림질 1kg당 작업시간(분)
IRON_BATCH_KG  = 4        # 다림질 배치 크기(kg)

res = solve_line(WASHERS, wash, dry, IRON_SETUP_MIN, IRON_PERKG_MIN, IRON_BATCH_KG)
show(res['table'])
res

,공정,설비대수,사이클(준비+작업),배치용량,처리량 kg/h,필요 인원
0,세탁,6대,10 + 50 (분),3 kg,18.00,1명
1,건조,3대,5 + 35 (분),4 kg,18.00,1명
2,다림질,-,준비 10 + 작업 10×배치kg (분),4 kg,4.80 (1인),4명


{'wash_tph': 18.0,
 'dryers': 3,
 'dry_tph': 18.0,
 'iron_tph_one': 4.8,
 'wash_people': 1,
 'dry_people': 1,
 'iron_people': 4,
 'table':     공정 설비대수              사이클(준비+작업)  배치용량   처리량 kg/h 필요 인원
 0   세탁   6대             10 + 50 (분)  3 kg      18.00    1명
 1   건조   3대              5 + 35 (분)  4 kg      18.00    1명
 2  다림질    -  준비 10 + 작업 10×배치kg (분)  4 kg  4.80 (1인)    4명}

### **시나리오 1) 세탁기 8대로 증설**
- 변경: `WASHERS = 8`

In [4]:
WASHERS_1 = 8
res1 = solve_line(WASHERS_1, wash, dry, IRON_SETUP_MIN, IRON_PERKG_MIN, IRON_BATCH_KG)
show(res1['table'])
res1

,공정,설비대수,사이클(준비+작업),배치용량,처리량 kg/h,필요 인원
0,세탁,8대,10 + 50 (분),3 kg,24.00,2명
1,건조,4대,5 + 35 (분),4 kg,24.00,1명
2,다림질,-,준비 10 + 작업 10×배치kg (분),4 kg,4.80 (1인),5명


{'wash_tph': 24.0,
 'dryers': 4,
 'dry_tph': 24.0,
 'iron_tph_one': 4.8,
 'wash_people': 2,
 'dry_people': 1,
 'iron_people': 5,
 'table':     공정 설비대수              사이클(준비+작업)  배치용량   처리량 kg/h 필요 인원
 0   세탁   8대             10 + 50 (분)  3 kg      24.00    2명
 1   건조   4대              5 + 35 (분)  4 kg      24.00    1명
 2  다림질    -  준비 10 + 작업 10×배치kg (분)  4 kg  4.80 (1인)    5명}

### **시나리오 2) 다림질 배치를 3kg로 변경**
- 변경: `IRON_BATCH_KG = 3` (세탁 배치와 동기화)

In [5]:
IRON_BATCH_KG_2 = 3
res2 = solve_line(WASHERS, wash, dry, IRON_SETUP_MIN, IRON_PERKG_MIN, IRON_BATCH_KG_2)
show(res2['table'])
res2

,공정,설비대수,사이클(준비+작업),배치용량,처리량 kg/h,필요 인원
0,세탁,6대,10 + 50 (분),3 kg,18.00,1명
1,건조,3대,5 + 35 (분),4 kg,18.00,1명
2,다림질,-,준비 10 + 작업 10×배치kg (분),3 kg,4.50 (1인),4명


{'wash_tph': 18.0,
 'dryers': 3,
 'dry_tph': 18.0,
 'iron_tph_one': 4.5,
 'wash_people': 1,
 'dry_people': 1,
 'iron_people': 4,
 'table':     공정 설비대수              사이클(준비+작업)  배치용량   처리량 kg/h 필요 인원
 0   세탁   6대             10 + 50 (분)  3 kg      18.00    1명
 1   건조   3대              5 + 35 (분)  4 kg      18.00    1명
 2  다림질    -  준비 10 + 작업 10×배치kg (분)  3 kg  4.50 (1인)    4명}

### **시나리오 3) 건조 단계 작업준비 3분으로 단축**
- 변경: `dry.setup_min = 3`

In [6]:
dry3 = StageParams(setup_min=3, process_min=dry.process_min, batch_kg=dry.batch_kg)
res3 = solve_line(WASHERS, wash, dry3, IRON_SETUP_MIN, IRON_PERKG_MIN, IRON_BATCH_KG)
show(res3['table'])
res3

,공정,설비대수,사이클(준비+작업),배치용량,처리량 kg/h,필요 인원
0,세탁,6대,10 + 50 (분),3 kg,18.00,1명
1,건조,3대,3 + 35 (분),4 kg,18.95,1명
2,다림질,-,준비 10 + 작업 10×배치kg (분),4 kg,4.80 (1인),4명


{'wash_tph': 18.0,
 'dryers': 3,
 'dry_tph': 18.947368421052634,
 'iron_tph_one': 4.8,
 'wash_people': 1,
 'dry_people': 1,
 'iron_people': 4,
 'table':     공정 설비대수              사이클(준비+작업)  배치용량   처리량 kg/h 필요 인원
 0   세탁   6대             10 + 50 (분)  3 kg      18.00    1명
 1   건조   3대              3 + 35 (분)  4 kg      18.95    1명
 2  다림질    -  준비 10 + 작업 10×배치kg (분)  4 kg  4.80 (1인)    4명}

### **결과 비교 요약**
세 가지 시나리오를 한 표로 비교합니다.

In [7]:
import pandas as pd
summary = pd.DataFrame([
    ["기본",      WASHERS, res['dryers'],  res['wash_tph'],  res['dry_tph'],  res['iron_tph_one'], res['wash_people'], res['dry_people'], res['iron_people']],
    ["세탁8대",    8,       res1['dryers'], res1['wash_tph'], res1['dry_tph'], res1['iron_tph_one'], res1['wash_people'], res1['dry_people'], res1['iron_people']],
    ["다림3kg",    WASHERS, res2['dryers'], res2['wash_tph'], res2['dry_tph'], res2['iron_tph_one'], res2['wash_people'], res2['dry_people'], res2['iron_people']],
    ["건조준비3분", WASHERS, res3['dryers'], res3['wash_tph'], res3['dry_tph'], res3['iron_tph_one'], res3['wash_people'], res3['dry_people'], res3['iron_people']],
], columns=["시나리오","세탁기(대)","건조기(대)","세탁 kg/h","건조 kg/h","다림 1인 kg/h","세탁 인원","건조 인원","다림 인원"])

show(summary)
summary

,시나리오,세탁기(대),건조기(대),세탁 kg/h,건조 kg/h,다림 1인 kg/h,세탁 인원,건조 인원,다림 인원
0,기본,6,3,18.0,18.000000,4.8,1,1,4
1,세탁8대,8,4,24.0,24.000000,4.8,2,1,5
2,다림3kg,6,3,18.0,18.000000,4.5,1,1,4
3,건조준비3분,6,3,18.0,18.947368,4.8,1,1,4


,시나리오,세탁기(대),건조기(대),세탁 kg/h,건조 kg/h,다림 1인 kg/h,세탁 인원,건조 인원,다림 인원
0,기본,6,3,18.0,18.000000,4.8,1,1,4
1,세탁8대,8,4,24.0,24.000000,4.8,2,1,5
2,다림3kg,6,3,18.0,18.000000,4.5,1,1,4
3,건조준비3분,6,3,18.0,18.947368,4.8,1,1,4




---

